In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────────────────
# Always run this first. If it says CPU, go to:
# Runtime → Change runtime type → T4 GPU → Save
# A T4 gives ~16GB VRAM, enough for EfficientNet-B0 with batch size 32.

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

if device.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU found. Switch runtime before continuing.")

In [ ]:
# ── Cell 2: Mount Google Drive and clone repo ─────────────────────────────────
# We mount Drive so we can:
#   1. Read the dataset (train/ and val/ folders you already have there)
#   2. Save checkpoints back to Drive so they survive a runtime restart
#
# Replace YOUR_GITHUB_USERNAME and YOUR_REPO_NAME with your actual values.
# If the repo is private, use a personal access token in the URL:
#   https://<token>@github.com/USERNAME/REPO.git

from google.colab import drive
drive.mount("/content/drive")

import os

REPO_URL  = "https://github.com/Lusine-Nasibyan/plant-disease-classifier.git"
REPO_NAME = "plant-disease-classifier"
REPO_PATH = f"/content/{REPO_NAME}"

if not os.path.exists(REPO_PATH):
    os.system(f"git clone {REPO_URL} {REPO_PATH}")
    print(f"Cloned → {REPO_PATH}")
else:
    # Pull latest code if already cloned (e.g. after a restart)
    os.system(f"cd {REPO_PATH} && git pull")
    print(f"Repo already exists, pulled latest.")

os.chdir(REPO_PATH)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# ── Cell 3: Install dependencies ─────────────────────────────────────────────
# Colab has torch pre-installed but we need timm, wandb, and scikit-learn.
# --quiet suppresses the long pip output so the cell stays readable.

os.system("pip install timm wandb scikit-learn --quiet")
print("Dependencies installed.")

In [ ]:
# ── Cell 4: Link dataset from Drive ──────────────────────────────────────────
# Instead of copying 8k images (slow), we symlink the Drive folders
# directly into the repo's data/raw/ directory.
# Update DRIVE_TRAIN and DRIVE_VAL to match where your data lives in Drive.
#
# Example: if your Drive has:
#   My Drive/plant-dataset/train/
#   My Drive/plant-dataset/val/
# then set:
#   DRIVE_TRAIN = "/content/drive/MyDrive/plant-dataset/train"
#   DRIVE_VAL   = "/content/drive/MyDrive/plant-dataset/val"

import os

DRIVE_TRAIN = "/content/drive/MyDrive/plant_disease_classifier/train"
DRIVE_VAL   = "/content/drive/MyDrive/plant_disease_classifier/val"

LOCAL_TRAIN = f"{REPO_PATH}/data/raw/train"
LOCAL_VAL   = f"{REPO_PATH}/data/raw/val"

os.makedirs(f"{REPO_PATH}/data/raw", exist_ok=True)
os.makedirs(f"{REPO_PATH}/data/processed", exist_ok=True)
os.makedirs(f"{REPO_PATH}/models", exist_ok=True)

# Create symlinks so the training code can find data at the expected paths
for src, dst in [(DRIVE_TRAIN, LOCAL_TRAIN), (DRIVE_VAL, LOCAL_VAL)]:
    if not os.path.exists(dst):
        os.symlink(src, dst)
        print(f"Linked: {dst} → {src}")
    else:
        print(f"Already exists: {dst}")

# Quick sanity check — count images through the symlink
train_count = sum(
    len(files)
    for _, _, files in os.walk(LOCAL_TRAIN)
)
val_count = sum(
    len(files)
    for _, _, files in os.walk(LOCAL_VAL)
)
print(f"\nTrain images visible : {train_count}")
print(f"Val   images visible : {val_count}")

In [ ]:
# ── Cell 5: Prepare data (build labels.csv) ───────────────────────────────────
# Runs prepare_data.py to produce data/processed/labels.csv.
# This is fast (<5 seconds) — it only reads folder names and writes a CSV.
# Skip this cell if labels.csv already exists in the repo.

import sys
sys.path.insert(0, f"{REPO_PATH}/src")

# Run as a module so relative paths resolve correctly
os.system(f"cd {REPO_PATH} && python src/prepare_data.py")

In [ ]:
# ── Cell 6: Log in to Weights & Biases ───────────────────────────────────────
# Paste your API key when prompted. Get it from https://wandb.ai/authorize
# All training runs will appear at https://wandb.ai/YOUR_USERNAME/plant-disease-classifier

import wandb
wandb.login()

In [ ]:
# ── Cell 7: Training utility ──────────────────────────────────────────────────
# Wraps train.py so we can launch runs with a simple function call
# rather than subprocess. This also means Python errors surface cleanly
# inside the notebook rather than being swallowed by a shell call.

import sys
sys.path.insert(0, f"{REPO_PATH}/src")

# Reload modules in case code changed since last cell run
import importlib
import dataset, model, train
importlib.reload(dataset)
importlib.reload(model)
importlib.reload(train)

from train import train, DEFAULT_CFG

def run_experiment(overrides: dict) -> None:
    """
    Launch a training run with DEFAULT_CFG plus any overrides.

    Args:
        overrides: dict of config keys to override, e.g.
            {"backbone": "mobilenetv3_small", "weighted_loss": False}
    """
    cfg = DEFAULT_CFG.copy()
    cfg.update(overrides)

    print("\n" + "="*60)
    print("  EXPERIMENT CONFIG")
    print("="*60)
    for k, v in cfg.items():
        print(f"  {k:<25} {v}")
    print("="*60 + "\n")

    train(cfg)

In [ ]:
# ── Cell 8: Primary run ───────────────────────────────────────────────────────
# EfficientNet-B0 with inverse-frequency weighted cross-entropy loss.
# This is the main model — everything else is an ablation of this.
#
# Expected time on Colab T4:
#   Stage 1 (5 epochs)  : ~4 minutes
#   Stage 2 (20 epochs) : ~16 minutes
#   Total               : ~20 minutes

run_experiment({
    "backbone"        : "efficientnet_b0",
    "weighted_loss"   : True,
    "stage1_epochs"   : 5,
    "stage2_epochs"   : 20,
})

In [ ]:
# ── Cell 9: Ablation 1 — plain cross-entropy ─────────────────────────────────
# Identical to Cell 8 except weighted_loss=False.
# Isolates the contribution of class weighting to final performance.
# Compare this run's val_acc and mAP to the primary run in W&B.

run_experiment({
    "backbone"      : "efficientnet_b0",
    "weighted_loss" : False,
    "stage1_epochs" : 5,
    "stage2_epochs" : 20,
})

In [ ]:
# ── Cell 10: Ablation 2 — weighted sampler ────────────────────────────────────
# Uses plain CE loss but oversamples minority classes at the data level.
# Comparing this to Cell 9 shows whether the correction is better applied
# at the loss level (Cell 8) or the data level (this cell).

run_experiment({
    "backbone"             : "efficientnet_b0",
    "weighted_loss"        : False,
    "use_weighted_sampler" : True,
    "stage1_epochs"        : 5,
    "stage2_epochs"        : 20,
})

In [ ]:
# ── Cell 11: Ablation 3 — MobileNetV3 backbone ───────────────────────────────
# Same training strategy as the primary run (weighted CE, two-stage)
# but with MobileNetV3-Small (~1.5M params vs EfficientNet's ~4M).
# This directly tests the accuracy-vs-efficiency tradeoff.

run_experiment({
    "backbone"      : "mobilenetv3_small",
    "weighted_loss" : True,
    "stage1_epochs" : 5,
    "stage2_epochs" : 20,
})

In [ ]:
# ── Cell 12: Evaluate all saved checkpoints ───────────────────────────────────
# Runs evaluate.py on every .pt file in models/ and prints mAP for each.
# Results are also saved as JSON files for building your report table.

import sys
sys.path.insert(0, f"{REPO_PATH}/src")

import importlib
import evaluate
importlib.reload(evaluate)

from evaluate import evaluate as run_eval
from pathlib import Path

checkpoints = sorted(Path(f"{REPO_PATH}/models").glob("*__best.pt"))

if not checkpoints:
    print("No checkpoints found. Run training cells first.")
else:
    results_summary = {}

    for ckpt in checkpoints:
        print(f"\n{'='*60}")
        print(f"  Evaluating: {ckpt.name}")
        print(f"{'='*60}")

        # Infer backbone from filename
        backbone = (
            "mobilenetv3_small"
            if "mobilenetv3" in ckpt.name
            else "efficientnet_b0"
        )

        cfg = {
            "checkpoint" : str(ckpt),
            "backbone"   : backbone,
            "class_map"  : f"{REPO_PATH}/models/class_mapping.json",
            "batch_size" : 32,
            "out_dir"    : f"{REPO_PATH}/models",
        }

        map_score = run_eval(cfg)
        results_summary[ckpt.name] = round(map_score, 4)

    # Print comparison table
    print(f"\n{'='*60}")
    print("  RESULTS SUMMARY")
    print(f"{'='*60}")
    print(f"  {'Checkpoint':<50} {'mAP':>6}")
    print("  " + "─"*56)
    for name, score in sorted(results_summary.items(), key=lambda x: -x[1]):
        print(f"  {name:<50} {score:.4f}")

In [ ]:
# ── Cell 13: Save best model to Drive ────────────────────────────────────────
# Colab's local storage is wiped when the runtime disconnects.
# This copies your best checkpoint and class mapping to Drive
# so they persist and can be downloaded for the API deployment.

import shutil
from pathlib import Path

DRIVE_SAVE = "/content/drive/MyDrive/plant_disease_classifier/saved_models"
os.makedirs(DRIVE_SAVE, exist_ok=True)

files_to_save = list(Path(f"{REPO_PATH}/models").glob("*.pt")) + \
                list(Path(f"{REPO_PATH}/models").glob("*.json"))

for f in files_to_save:
    dest = f"{DRIVE_SAVE}/{f.name}"
    shutil.copy(f, dest)
    print(f"Saved → {dest}")